# Immigration Case Triage Assistant
### AI-Assisted Legal Workflow Prototype

**Created by Frances V. Perez, Esq.**

This project explores how generative AI can support the initial organization and triage of immigration client intake information while preserving attorney judgment and human review.

The prototype converts an unstructured immigration intake narrative into
structured case information, validates the AI-generated output, identifies
missing or conflicting facts, applies deterministic triage rules, and
generates an attorney-facing review report.

## System Architecture

**Client Intake Narrative**  
↓  
**AI Fact Extraction**  
↓  
**Structured Case Data**  
↓  
**Schema & Type Validation**  
↓  
**Deterministic Triage Rules**  
↓  
**Attorney Review Report**  
↓  
**Human Attorney Judgment**

## Design Principles

- AI is used for language understanding and fact extraction.
- Deterministic Python rules are used for predictable workflow logic.
- Unknown information is preserved rather than guessed.
- Contradictory facts are surfaced rather than resolved by the model.
- Malformed AI output is stopped before downstream processing.
- The system identifies issues for attorney review rather than making
  legal eligibility determinations.
- Human attorney judgment remains the final decision-making layer.

## Evaluation

Prompt iterations were tested against synthetic immigration intake
scenarios designed to expose uncertainty, contradictions, entity
attribution problems, incomplete dates, unsupported inference, and
structured-output failures.

During development regression testing, Prompt V5 produced 63/64 correct factual fields (98.4%) across Test Cases 3–6. A later validation run of the finalized prototype produced 64/64 factual fields (100.0%) on the same small synthetic test set.

These results are run-specific experimental benchmarks and should not be interpreted as general accuracy on immigration matters or real-world client files.

## Data & Confidentiality

All client scenarios used in this project are synthetic. No confidential
client information is included.

# 1. Problem and Design Approach

Legal client intake narratives often contain incomplete, uncertain, or
contradictory information. A useful legal-AI workflow should organize that
information without silently converting uncertainty into fact or replacing
attorney judgment.

This prototype separates responsibilities across three layers:

1. **AI extraction:** Converts unstructured intake language into structured facts.
2. **Deterministic workflow logic:** Validates structured output and identifies
   predefined issues requiring review.
3. **Attorney judgment:** Reviews missing information, factual conflicts, and
   flagged issues before making any legal determination.

This separation was intentional. Generative AI is used where language
interpretation is useful, while predictable Python logic is used where
consistent workflow behavior is preferable.

# 2. System Architecture

The prototype uses a human-in-the-loop workflow designed to separate
probabilistic AI tasks from deterministic processing.

### Workflow

**Unstructured Client Intake**  
↓  
**AI Fact Extraction (Prompt V5)**  
↓  
**Structured JSON**  
↓  
**Schema & Data-Type Validation**  
↓  
**Deterministic Triage Rules**  
↓  
**Attorney-Facing Report**  
↓  
**Human Attorney Review**

### Component Responsibilities

**AI Layer**
- Extracts explicitly stated facts from narrative text.
- Preserves unknown information as `null`.
- Identifies contradictory statements.
- Normalizes sufficiently precise dates.
- Avoids making legal eligibility determinations.

**Validation Layer**
- Confirms that all required fields are present.
- Checks expected data types.
- Stops malformed output before downstream processing.

**Deterministic Triage Layer**
- Identifies predefined missing information.
- Applies rule-based attorney-review flags.
- Preserves factual conflicts identified during extraction.
- Does not independently determine eligibility for relief.

**Presentation Layer**
- Converts machine-readable values into attorney-friendly language.
- Distinguishes missing information, factual conflicts, and review flags.
- Produces a standardized attorney triage report.

### Human Review

The system is designed as a decision-support workflow, not an autonomous
legal decision-maker. Attorney review remains necessary before any legal
conclusion, filing decision, or case strategy is made.

# 3. AI Fact Extraction

The AI component converts an unstructured client intake narrative into a
defined structured schema.

The extraction prompt was iteratively refined after testing revealed
specific failure modes. The final prompt used in the prototype is Prompt V5.

### Extraction Requirements

The model is instructed to:

- extract only explicitly and unambiguously stated facts
- preserve missing or uncertain information as `null`
- avoid converting uncertain language into definite facts
- identify contradictory statements without resolving them
- keep facts attributed to the correct individual
- normalize sufficiently precise dates to `YYYY-MM-DD`
- avoid unsupported asylum-filing-timing inferences
- return structured JSON
- avoid legal advice and eligibility determinations

This conservative extraction policy intentionally favors abstention over
unsupported factual inference.

# 4. Validation and Processing Controls

AI-generated structured data is not automatically trusted by the workflow.

Before extracted information can reach the deterministic triage layer,
the output passes through a validation gate.

### Required-Field Validation

The system checks whether every field required by the downstream workflow
is present in the AI-generated output.

A missing field causes validation to fail.

### Data-Type Validation

The system also verifies that each value has an expected data type.

Examples include:

- `age` → integer or `null`
- `in_removal_proceedings` → boolean or `null`
- `i360_approved` → boolean or `null`
- `entry_date` → string or `null`
- `conflicts` → list

Allowing `null` is intentional. Unknown information should remain unknown
rather than being converted into an unsupported `true` or `false` value.

### Processing Gate

If required fields are missing or values have invalid data types, processing
stops before the output reaches the triage rules.

This creates the following control:

**AI Output**  
↓  
**Schema & Type Validation**  
↓  
**PASS → Continue to Triage**  
**FAIL → Stop for Review**

The validation layer does not establish factual accuracy. Factual accuracy is
evaluated separately against attorney-defined benchmark outputs.

# 5. Deterministic Attorney-Triage Logic

After AI-generated case data passes validation, the structured information
is processed through deterministic Python rules.

The purpose of this layer is not to determine eligibility for immigration
relief. It identifies predefined information gaps and factual combinations
that should be surfaced for attorney review.

### Missing Information

The workflow identifies information that may require follow-up during
case intake, including:

- manner of entry
- next immigration court hearing date

### Attorney Review Flags

The prototype contains example rules that identify factual combinations
requiring additional attorney analysis.

Examples include:

- approved I-360 + removal proceedings
- asylum filed + extracted one-year timing issue
- approved I-360 + I-485 not filed

These flags are intentionally phrased as issues for review rather than
legal conclusions.

### Factual Conflicts

Contradictions identified during AI extraction are preserved separately
from attorney-review flags.

This distinction allows the report to differentiate among:

**Missing Information** — a fact has not been established.

**Factual Conflict** — the intake contains competing statements.

**Attorney Review Flag** — established structured facts trigger a
predefined workflow rule.

### Design Rationale

Generative AI is used to interpret narrative language, while deterministic
code handles predefined workflow rules.

This separation improves transparency because each attorney-review flag
can be traced to an explicit rule rather than an unexplained model conclusion.

# 6. Evaluation Methodology

The extraction prompt was evaluated using synthetic immigration intake
scenarios with attorney-defined expected outputs.

Each test case was designed to expose a different potential legal-AI
failure mode.

### Test Scenarios

**Test 3 — Contradictory Facts**
- conflicting entry dates
- conflicting I-360 information
- uncertainty regarding removal proceedings
- tested whether the model preserved contradictions rather than resolving them

**Test 4 — Multiple Individuals**
- facts concerning the client and another individual appeared in the same narrative
- tested entity attribution and date normalization
- tested whether unrelated facts would be incorrectly assigned to the client

**Test 5 — Noisy Narrative**
- relevant facts were mixed with distracting information
- asylum filing timing was incomplete
- tested unsupported inference and incomplete-date handling

**Test 6 — Sparse and Uncertain Intake**
- numerous facts were unknown or expressed with uncertainty
- tested whether the model would preserve unknown values rather than invent information
- served as an unseen test case for Prompt V5

### Evaluation Method

For each scenario, an attorney-defined expected structured output was created.

The model output was then compared field-by-field against that benchmark.

The evaluation tracked:

- correct factual fields
- incorrect or unsupported factual fields
- missing facts
- uncertainty handling
- conflict preservation
- regressions introduced by prompt changes

Conflict descriptions were evaluated separately because equivalent factual
conflicts can be expressed using different natural-language wording.

### Development Evaluation Result

During regression testing, Prompt V5 produced:

63 correct factual fields out of 64 evaluated fields — 98.4%

This result reflects the development-stage regression evaluation and preserves the observed Test 3 disagreement documented in Section 7. The disagreement was retained rather than changing the attorney-defined benchmark after observing the model output.

The unseen sparse-intake Test Case 6 produced:

16/16 correct factual fields

A later execution of the finalized portfolio evaluation produced 64/64 factual fields (100.0%). Because generative AI output can vary between runs, both results are reported rather than replacing the earlier regression result with the later score.

These results apply only to this small synthetic evaluation set and should not be interpreted as general accuracy on immigration matters or real-world client files.

# 7. Prompt Iteration and Failure Analysis

The extraction prompt was developed iteratively. Changes were made in
response to observed model errors rather than added only as theoretical
instructions.

### Key Iterations

**Uncertainty Handling**

Early testing showed that uncertain language could be converted into
definite structured facts.

The prompt was revised to require expressions such as "believes,"
"may have," "approximately," and similar uncertainty to remain `null`
unless the fact was otherwise clearly established.

---

**Contradiction Preservation**

Testing introduced narratives containing competing factual statements.

The prompt was revised so that the model would:

- set the disputed structured field to `null`
- preserve the competing statements in a `conflicts` list
- avoid deciding which version was correct

---

**Entity Attribution**

A test narrative contained facts about both the client and another person.

Prompt V3 produced **13/16 correct factual fields (81.2%)** on this test.

After adding more explicit entity-attribution, date-formatting, and asylum
timing instructions, Prompt V4 produced **16/16 (100%)** on the same case.

---

**Incomplete Asylum Filing Timing**

A later test stated that an asylum application was filed "sometime in 2025"
without providing a complete filing date.

Prompt V4 produced **15/16 correct factual fields (93.8%)** because it made
an unsupported timing inference.

Prompt V5 added a rule requiring sufficiently precise timing information
before populating the asylum timing field.

On the same test, Prompt V5 produced **16/16 (100%)**.

---

### Regression Testing

Prompt changes were also tested against earlier scenarios.

Prompt V5 produced:

- Test 3: **15/16**
- Test 4: **16/16**
- Test 5: **16/16**
- Test 6: **16/16**

This regression testing revealed one remaining disagreement involving
`i360_filed`.

The model returned `true` where the attorney-defined benchmark required
`null`. Although other facts in the narrative logically implied that an
I-360 existed, the project's conservative extraction policy required the
filing fact to be explicitly stated.

This error was retained in the evaluation rather than changing the benchmark
to improve the reported score.

### Takeaway

Prompt improvements can solve one failure mode while introducing or exposing
another. For that reason, prompt changes should be evaluated against both the
new scenario and previously tested scenarios.

# 8. Attorney-Facing Prototype

The final prototype combines the extraction, validation, triage, and
reporting components into a single workflow.

### End-to-End Processing

When an intake narrative is submitted, the application:

1. Sends the narrative to the AI extraction layer.
2. Converts the response into structured case data.
3. Validates required fields and data types.
4. Stops processing if the structured output fails validation.
5. Applies deterministic triage rules to valid data.
6. Preserves missing information and factual conflicts.
7. Generates a standardized attorney-facing report.

### Attorney Triage Report

The final report organizes information into:

- client information
- immigration court information
- SIJS / I-360 information
- asylum information
- adjustment-of-status information
- missing information
- factual conflicts
- attorney-review flags

Boolean and missing values are translated into attorney-friendly
**Yes / No / Unknown** language rather than exposing raw Python values.

### Interactive Interface

A lightweight Gradio interface was added to demonstrate how the workflow
could be presented to an end user.

The interface accepts a synthetic immigration intake narrative and returns
the completed attorney triage report.

The interface is a prototype for workflow demonstration only and is not
intended for processing confidential client information.

# 9. Limitations and Future Development

This project is an experimental legal-AI workflow prototype and is not
intended for production use.

### Current Limitations

- Evaluation was conducted on a small synthetic test set.
- The 98.4% factual-field result does not establish general model accuracy.
- The prototype does not independently verify whether extracted client
  statements are factually true.
- The current validation layer checks required fields and data types but
  does not perform comprehensive semantic validation.
- The deterministic triage rules represent a limited set of example
  immigration workflow issues.
- AI behavior may vary across model versions and repeated runs.
- The prototype has not undergone production security, privacy, or
  confidentiality review.

### Potential Future Improvements

Future development could include:

- API-level structured-output enforcement using a formal JSON schema
- expanded synthetic evaluation datasets
- automated regression testing
- date-format and semantic validation
- confidence or uncertainty tracking
- source attribution linking extracted facts to intake text
- broader immigration workflow rules
- model-output monitoring and error logging
- user feedback mechanisms for attorney corrections
- secure authentication and access controls
- privacy and confidentiality safeguards appropriate for legal data

### Human-in-the-Loop Requirement

The system is designed to assist with information organization and issue
spotting, not to replace professional legal judgment.

Any production implementation involving real client information would
require additional testing, security controls, confidentiality safeguards,
and attorney oversight.

### What This Prototype Demonstrates

This project demonstrates an applied legal-engineering approach to generative AI:

- translating a legal workflow into a structured technical process
- designing and iterating prompts based on observed model failures
- creating attorney-defined benchmarks for AI evaluation
- testing uncertainty, contradictions, entity attribution, and unsupported inference
- separating probabilistic AI extraction from deterministic workflow logic
- implementing validation and processing safeguards
- conducting regression testing after prompt changes
- designing a human-in-the-loop workflow that preserves attorney judgment

The project was developed from the perspective of a practicing immigration attorney exploring how AI can support legal work while maintaining appropriate safeguards and human oversight.

# 10. Prototype Implementation

The following cells contain the streamlined implementation of the final prototype, from AI-assisted fact extraction through validation, deterministic triage, reporting, and the interactive attorney-facing interface.

## Setup

The prototype uses the OpenAI API for structured fact extraction and
Gradio for the interactive demonstration.

The API credential is loaded from Google Colab Secrets and is not stored
directly in the notebook. To run the prototype, a user must add their own OpenAI API key to Google Colab Secrets using the name OPENAI_API_KEY. The notebook does not contain or expose an API credential.

In [63]:
# Install required packages
!pip -q install openai gradio

# Imports
import json
import gradio as gr
from openai import OpenAI
from google.colab import userdata

# Load API key securely from Google Colab Secrets
api_key = userdata.get("OPENAI_API_KEY")
client_ai = OpenAI(api_key=api_key)

print("Setup complete.")

Setup complete.


## AI Fact-Extraction Prompt

Prompt V5 reflects the final prompt iteration used in the prototype.
It incorporates rules developed through testing for uncertainty,
contradictions, entity attribution, date normalization, and unsupported
factual inference.

In [64]:
extraction_prompt_v5 = """
You are assisting an immigration attorney with factual intake organization.

Your task is FACT EXTRACTION ONLY.

Do not provide legal advice.
Do not determine eligibility for immigration relief.
Do not make legal conclusions.
Do not fill factual gaps using assumptions or logical inference.

Extract only facts that are explicitly and unambiguously stated in the
client intake narrative.

Return ONLY valid JSON using exactly these fields:

{
  "name": null,
  "age": null,
  "country": null,
  "entry_date": null,
  "manner_of_entry": null,
  "in_removal_proceedings": null,
  "immigration_court": null,
  "next_hearing": null,
  "family_court_order": null,
  "i360_filed": null,
  "i360_approved": null,
  "i360_priority_date": null,
  "deferred_action": null,
  "asylum_filed": null,
  "asylum_within_one_year": null,
  "i485_filed": null,
  "conflicts": []
}

EXTRACTION RULES:

1. Use null when information is missing, uncertain, ambiguous, incomplete,
   or not explicitly stated.

2. Do not infer a fact merely because another fact logically suggests it.

3. Treat uncertain language conservatively. Statements using language such as
   "believes," "thinks," "may have," "might have," "approximately,"
   "probably," or "not sure" should remain null unless the fact is otherwise
   independently and clearly established in the narrative.

4. If the narrative contains conflicting statements about the same fact:
   - set the disputed field to null;
   - describe the competing statements in the "conflicts" list;
   - do not decide which statement is correct.

5. Attribute facts only to the client. Do not assign another person's
   immigration history, dates, applications, court information, or status
   to the client.

6. When a complete date is explicitly known, format it as YYYY-MM-DD.
   If the date is incomplete or uncertain and the schema requires a complete
   date, use null.

7. "asylum_within_one_year" may be true or false only when:
   - asylum_filed is explicitly established as true; and
   - the narrative contains sufficiently precise entry and asylum filing
     timing to make the factual timing comparison.

8. Vague or incomplete asylum filing timing is insufficient for
   "asylum_within_one_year." Examples such as "sometime in 2025,"
   "around June," or "approximately one year later" should result in null
   unless sufficiently precise dates are otherwise established.

9. Do not determine whether any legal exception to the asylum one-year
   filing deadline applies.

10. Return JSON only. Do not include commentary before or after the JSON.
"""

## Prompt Evaluation

The final extraction prompt was evaluated against attorney-defined benchmark
outputs using synthetic immigration intake scenarios.

The evaluation compares extracted factual fields against expected values
field-by-field. This provides a reproducible way to identify unsupported
inferences, missing facts, regressions, and uncertainty-handling errors.

The compact evaluation below preserves the final Test Cases 3–6 used to
calculate the reported benchmark:

**64 / 64 factual fields correct (100.0%)**

The benchmark reflects this small synthetic test set only and is not a
measure of general accuracy on immigration matters.

In [65]:
def evaluate_case(test_name, expected_output, actual_output):
    correct = 0
    total = 0
    failed_fields = []

    print(test_name)
    print("-" * len(test_name))

    for field in expected_output:
        # Conflict wording is evaluated separately because equivalent
        # conflicts may be expressed differently in natural language.
        if field == "conflicts":
            continue

        expected = expected_output[field]
        actual = actual_output.get(field)

        total += 1

        if expected == actual:
            print(f"{field}: PASS")
            correct += 1
        else:
            print(
                f"{field}: FAIL | "
                f"Expected: {expected} | Actual: {actual}"
            )
            failed_fields.append(field)

    accuracy = (correct / total) * 100

    print("\nRESULT")
    print("------")
    print(f"Correct factual fields: {correct}/{total}")
    print(f"Accuracy: {accuracy:.1f}%")

    if failed_fields:
        print("Failed fields:", failed_fields)
    else:
        print("Failed fields: None")

    return {
        "test_name": test_name,
        "correct": correct,
        "total": total,
        "accuracy": accuracy,
        "failed_fields": failed_fields
    }

### Synthetic Evaluation Cases

The following synthetic scenarios test the extraction system against deliberately difficult intake patterns. Each scenario has an attorney-defined expected output that serves as the benchmark for evaluating the model's factual extraction.

In [66]:
# Test Case 3 — Contradictory Facts

test_intake_3 = """
The client is a 20-year-old citizen of Ecuador.

During the initial intake, the client stated that he entered the
United States on March 15, 2023. Later in the same interview, he
stated that he entered on May 2, 2023.

The client stated that he is currently in removal proceedings before
the New York Immigration Court. He later stated that he believes his
immigration court case may have been dismissed, but he is not sure.

The client initially stated that USCIS approved his Form I-360.
Later, he stated that his attorney told him the I-360 was still pending.

The client has a Family Court order containing SIJS findings.

The client states that he has never filed Form I-589.

The client does not know whether Form I-485 has been filed on his behalf.
"""

expected_output_3 = {
    "name": None,
    "age": 20,
    "country": "Ecuador",
    "entry_date": None,
    "manner_of_entry": None,
    "in_removal_proceedings": None,
    "immigration_court": "New York Immigration Court",
    "next_hearing": None,
    "family_court_order": True,
    "i360_filed": None,
    "i360_approved": None,
    "i360_priority_date": None,
    "deferred_action": None,
    "asylum_filed": False,
    "asylum_within_one_year": None,
    "i485_filed": None,
    "conflicts": [
        "Conflicting entry dates",
        "Conflicting I-360 status",
        "Uncertain removal-proceedings status"
    ]
}

In [67]:
# Run Prompt V5 on Test Case 3

response_3 = client_ai.responses.create(
    model="gpt-5.6-luna",
    input=[
        {
            "role": "system",
            "content": extraction_prompt_v5
        },
        {
            "role": "user",
            "content": test_intake_3
        }
    ]
)

actual_output_3 = json.loads(response_3.output_text)

result_3 = evaluate_case(
    "Test Case 3 — Contradictory Facts",
    expected_output_3,
    actual_output_3
)

Test Case 3 — Contradictory Facts
---------------------------------
name: PASS
age: PASS
country: PASS
entry_date: PASS
manner_of_entry: PASS
in_removal_proceedings: PASS
immigration_court: PASS
next_hearing: PASS
family_court_order: PASS
i360_filed: FAIL | Expected: None | Actual: True
i360_approved: PASS
i360_priority_date: PASS
deferred_action: PASS
asylum_filed: PASS
asylum_within_one_year: PASS
i485_filed: PASS

RESULT
------
Correct factual fields: 15/16
Accuracy: 93.8%
Failed fields: ['i360_filed']


In [68]:
# Evaluate conflict detection separately from factual-field accuracy.

expected_conflict_count_3 = 3
actual_conflict_count_3 = len(actual_output_3["conflicts"])

print("CONFLICT DETECTION")
print("------------------")
print("Expected conflicts:", expected_conflict_count_3)
print("Detected conflicts:", actual_conflict_count_3)

if actual_conflict_count_3 == expected_conflict_count_3:
    print("Result: PASS")
else:
    print("Result: REVIEW REQUIRED")

CONFLICT DETECTION
------------------
Expected conflicts: 3
Detected conflicts: 3
Result: PASS


In [69]:
# Test Case 4 — Multiple Individuals / Entity Attribution

test_intake_4 = """
The client is Daniel, an 18-year-old citizen of Ecuador.

Daniel entered the United States on June 10, 2024. His manner of entry
is not stated.

Daniel is currently in removal proceedings before the New York
Immigration Court. He does not know his next hearing date.

Daniel's brother Mateo also has an immigration case, but Mateo entered
the United States on August 3, 2022 and his case is before a different
immigration court.

A Family Court issued an order containing SIJS findings for Daniel.

Daniel's attorney filed Form I-360. USCIS approved Daniel's I-360 with
a priority date of February 14, 2025.

Daniel has not filed an asylum application.

Daniel has not filed Form I-485.
"""

expected_output_4 = {
    "name": "Daniel",
    "age": 18,
    "country": "Ecuador",
    "entry_date": "2024-06-10",
    "manner_of_entry": None,
    "in_removal_proceedings": True,
    "immigration_court": "New York Immigration Court",
    "next_hearing": None,
    "family_court_order": True,
    "i360_filed": True,
    "i360_approved": True,
    "i360_priority_date": "2025-02-14",
    "deferred_action": None,
    "asylum_filed": False,
    "asylum_within_one_year": None,
    "i485_filed": False,
    "conflicts": []
}

In [70]:
# Run Prompt V5 on Test Case 4

response_4 = client_ai.responses.create(
    model="gpt-5.6-luna",
    input=[
        {
            "role": "system",
            "content": extraction_prompt_v5
        },
        {
            "role": "user",
            "content": test_intake_4
        }
    ]
)

actual_output_4 = json.loads(response_4.output_text)

result_4 = evaluate_case(
    "Test Case 4 — Multiple Individuals / Entity Attribution",
    expected_output_4,
    actual_output_4
)

Test Case 4 — Multiple Individuals / Entity Attribution
-------------------------------------------------------
name: PASS
age: PASS
country: PASS
entry_date: PASS
manner_of_entry: PASS
in_removal_proceedings: PASS
immigration_court: PASS
next_hearing: PASS
family_court_order: PASS
i360_filed: PASS
i360_approved: PASS
i360_priority_date: PASS
deferred_action: PASS
asylum_filed: PASS
asylum_within_one_year: PASS
i485_filed: PASS

RESULT
------
Correct factual fields: 16/16
Accuracy: 100.0%
Failed fields: None


In [71]:
# Test Case 5 — Noisy Narrative / Incomplete Timing

test_intake_5 = """
The client is Sofia, a 19-year-old citizen of Ecuador.

Sofia entered the United States on September 12, 2023. She does not
know the location where she crossed into the United States.

She is currently in removal proceedings before the New York Immigration
Court but does not know her next hearing date.

Sofia lives with her aunt. Her cousin also has an immigration case.

A Family Court issued an order containing SIJS findings for Sofia.
Her attorney filed Form I-360. USCIS approved the petition on
March 5, 2025, with a priority date of December 18, 2024.

USCIS also granted Sofia deferred action.

Sofia filed Form I-589 sometime in 2025, but she does not remember
the month or day when it was filed.

Sofia has not filed Form I-485.
"""

expected_output_5 = {
    "name": "Sofia",
    "age": 19,
    "country": "Ecuador",
    "entry_date": "2023-09-12",
    "manner_of_entry": None,
    "in_removal_proceedings": True,
    "immigration_court": "New York Immigration Court",
    "next_hearing": None,
    "family_court_order": True,
    "i360_filed": True,
    "i360_approved": True,
    "i360_priority_date": "2024-12-18",
    "deferred_action": True,
    "asylum_filed": True,
    "asylum_within_one_year": None,
    "i485_filed": False,
    "conflicts": []
}

In [72]:
# Run Prompt V5 on Test Case 5

response_5 = client_ai.responses.create(
    model="gpt-5.6-luna",
    input=[
        {
            "role": "system",
            "content": extraction_prompt_v5
        },
        {
            "role": "user",
            "content": test_intake_5
        }
    ]
)

actual_output_5 = json.loads(response_5.output_text)

result_5 = evaluate_case(
    "Test Case 5 — Noisy Narrative / Incomplete Timing",
    expected_output_5,
    actual_output_5
)

Test Case 5 — Noisy Narrative / Incomplete Timing
-------------------------------------------------
name: PASS
age: PASS
country: PASS
entry_date: PASS
manner_of_entry: PASS
in_removal_proceedings: PASS
immigration_court: PASS
next_hearing: PASS
family_court_order: PASS
i360_filed: PASS
i360_approved: PASS
i360_priority_date: PASS
deferred_action: PASS
asylum_filed: PASS
asylum_within_one_year: PASS
i485_filed: PASS

RESULT
------
Correct factual fields: 16/16
Accuracy: 100.0%
Failed fields: None


In [73]:
# Test Case 6 — Sparse / Uncertain Intake

test_intake_6 = """
The client is a 20-year-old citizen of Guatemala.

She came to the United States several years ago but does not remember
the exact date of entry.

She believes she may have attended an immigration court hearing in
the past, but she does not know whether she is currently in removal
proceedings or which immigration court handled the case.

The client remembers signing immigration paperwork with a previous
attorney but does not know what forms were filed.

She has heard the term SIJS but does not know whether a Family Court
order was ever issued or whether an I-360 was filed.

She is not sure whether an asylum application was filed on her behalf.

The client does not know whether Form I-485 has ever been filed.
"""

expected_output_6 = {
    "name": None,
    "age": 20,
    "country": "Guatemala",
    "entry_date": None,
    "manner_of_entry": None,
    "in_removal_proceedings": None,
    "immigration_court": None,
    "next_hearing": None,
    "family_court_order": None,
    "i360_filed": None,
    "i360_approved": None,
    "i360_priority_date": None,
    "deferred_action": None,
    "asylum_filed": None,
    "asylum_within_one_year": None,
    "i485_filed": None,
    "conflicts": []
}

In [74]:
response_6 = client_ai.responses.create(
    model="gpt-5.6-luna",
    input=[
        {
            "role": "system",
            "content": extraction_prompt_v5
        },
        {
            "role": "user",
            "content": test_intake_6
        }
    ]
)

actual_output_6 = json.loads(response_6.output_text)

result_6 = evaluate_case(
    "Test Case 6 — Sparse / Uncertain Intake",
    expected_output_6,
    actual_output_6
)

Test Case 6 — Sparse / Uncertain Intake
---------------------------------------
name: PASS
age: PASS
country: PASS
entry_date: PASS
manner_of_entry: PASS
in_removal_proceedings: PASS
immigration_court: PASS
next_hearing: PASS
family_court_order: PASS
i360_filed: PASS
i360_approved: PASS
i360_priority_date: PASS
deferred_action: PASS
asylum_filed: PASS
asylum_within_one_year: PASS
i485_filed: PASS

RESULT
------
Correct factual fields: 16/16
Accuracy: 100.0%
Failed fields: None


In [75]:
# Combined evaluation summary — Tests 3 through 6

evaluation_results = [
    result_3,
    result_4,
    result_5,
    result_6
]

total_correct = sum(
    result["correct"] for result in evaluation_results
)

total_fields = sum(
    result["total"] for result in evaluation_results
)

overall_accuracy = (total_correct / total_fields) * 100

print("FINAL EVALUATION SUMMARY")
print("------------------------")

for result in evaluation_results:
    print(
        f"{result['test_name']}: "
        f"{result['correct']}/{result['total']} "
        f"({result['accuracy']:.1f}%)"
    )

print("\nCombined factual fields:")
print(f"{total_correct}/{total_fields}")

print(f"\nOverall factual-field accuracy: {overall_accuracy:.1f}%")

print(
    "\nNote: Results apply only to this small synthetic "
    "evaluation set and do not represent general model accuracy."
)

FINAL EVALUATION SUMMARY
------------------------
Test Case 3 — Contradictory Facts: 15/16 (93.8%)
Test Case 4 — Multiple Individuals / Entity Attribution: 16/16 (100.0%)
Test Case 5 — Noisy Narrative / Incomplete Timing: 16/16 (100.0%)
Test Case 6 — Sparse / Uncertain Intake: 16/16 (100.0%)

Combined factual fields:
63/64

Overall factual-field accuracy: 98.4%

Note: Results apply only to this small synthetic evaluation set and do not represent general model accuracy.


## Output Validation

Before AI-generated data can enter the triage workflow, the prototype
checks that the expected fields are present and that each value has the
expected data type.

This validation gate prevents malformed structured output from being
silently passed into downstream processing.

`null` values are permitted because unknown information is intentionally
preserved rather than guessed.

In [76]:
required_fields = [
    "name",
    "age",
    "country",
    "entry_date",
    "manner_of_entry",
    "in_removal_proceedings",
    "immigration_court",
    "next_hearing",
    "family_court_order",
    "i360_filed",
    "i360_approved",
    "i360_priority_date",
    "deferred_action",
    "asylum_filed",
    "asylum_within_one_year",
    "i485_filed",
    "conflicts"
]

expected_types = {
    "name": (str, type(None)),
    "age": (int, type(None)),
    "country": (str, type(None)),
    "entry_date": (str, type(None)),
    "manner_of_entry": (str, type(None)),
    "in_removal_proceedings": (bool, type(None)),
    "immigration_court": (str, type(None)),
    "next_hearing": (str, type(None)),
    "family_court_order": (bool, type(None)),
    "i360_filed": (bool, type(None)),
    "i360_approved": (bool, type(None)),
    "i360_priority_date": (str, type(None)),
    "deferred_action": (bool, type(None)),
    "asylum_filed": (bool, type(None)),
    "asylum_within_one_year": (bool, type(None)),
    "i485_filed": (bool, type(None)),
    "conflicts": (list,)
}


def validate_output(output):
    errors = []

    # Confirm the AI returned a JSON object.
    if not isinstance(output, dict):
        return False, ["Output is not a JSON object."]

    # Check for required fields.
    for field in required_fields:
        if field not in output:
            errors.append(f"Missing required field: {field}")

    # Check data types for fields that are present.
    for field, allowed_types in expected_types.items():
        if field in output and not isinstance(output[field], allowed_types):
            errors.append(
                f"Invalid type for {field}: "
                f"{type(output[field]).__name__}"
            )

    if errors:
        return False, errors

    return True, []

## Deterministic Triage Logic

After the AI-generated data passes validation, Python applies a limited
set of predefined workflow rules.

These rules identify missing information and factual combinations that
should be surfaced for attorney review.

The rules do not determine eligibility for immigration relief. Their
purpose is to make the workflow predictable, transparent, and auditable.

In [77]:
def run_triage(client):
    missing_information = []
    review_flags = []

    # Identify important information that remains unknown.
    if client["manner_of_entry"] is None:
        missing_information.append("Manner of entry")

    if client["next_hearing"] is None:
        missing_information.append("Next immigration court hearing")

    # Apply predefined attorney-review rules.
    if (
        client["i360_approved"] is True
        and client["in_removal_proceedings"] is True
    ):
        review_flags.append(
            "Approved I-360 + removal proceedings: "
            "review procedural strategy."
        )

    if (
        client["asylum_filed"] is True
        and client["asylum_within_one_year"] is False
    ):
        review_flags.append(
            "Potential asylum one-year filing issue: "
            "attorney review required."
        )

    if (
        client["i360_approved"] is True
        and client["i485_filed"] is False
    ):
        review_flags.append(
            "I-360 approved but I-485 not filed: "
            "review adjustment eligibility and visa availability."
        )

    return {
        "missing_information": missing_information,
        "review_flags": review_flags,
        "conflicts": client["conflicts"]
    }

## Attorney-Facing Report

After validation and deterministic triage, the structured case information
is converted into a standardized report for attorney review.

The report separates:

- extracted client and case facts
- missing information
- factual conflicts
- attorney-review flags

Raw Python values are translated into attorney-friendly
**Yes / No / Unknown** language.

In [78]:
def display_value(value):
    if value is True:
        return "Yes"
    if value is False:
        return "No"
    if value is None:
        return "Unknown"
    return value


def build_report_text(client, triage_result):
    lines = []

    lines.append("IMMIGRATION CASE TRIAGE REPORT")
    lines.append("=" * 32)

    lines.append("\nCLIENT INFORMATION")
    lines.append(f"Name: {display_value(client['name'])}")
    lines.append(f"Age: {display_value(client['age'])}")
    lines.append(f"Country: {display_value(client['country'])}")
    lines.append(f"Entry Date: {display_value(client['entry_date'])}")
    lines.append(
        f"Manner of Entry: {display_value(client['manner_of_entry'])}"
    )

    lines.append("\nIMMIGRATION COURT")
    lines.append(
        "In Removal Proceedings: "
        f"{display_value(client['in_removal_proceedings'])}"
    )
    lines.append(
        f"Immigration Court: {display_value(client['immigration_court'])}"
    )
    lines.append(
        f"Next Hearing: {display_value(client['next_hearing'])}"
    )

    lines.append("\nSIJS / I-360")
    lines.append(
        f"Family Court Order: {display_value(client['family_court_order'])}"
    )
    lines.append(
        f"I-360 Filed: {display_value(client['i360_filed'])}"
    )
    lines.append(
        f"I-360 Approved: {display_value(client['i360_approved'])}"
    )
    lines.append(
        "I-360 Priority Date: "
        f"{display_value(client['i360_priority_date'])}"
    )
    lines.append(
        f"Deferred Action: {display_value(client['deferred_action'])}"
    )

    lines.append("\nASYLUM")
    lines.append(
        f"Asylum Filed: {display_value(client['asylum_filed'])}"
    )
    lines.append(
        "Filed Within One Year: "
        f"{display_value(client['asylum_within_one_year'])}"
    )

    lines.append("\nADJUSTMENT OF STATUS")
    lines.append(
        f"I-485 Filed: {display_value(client['i485_filed'])}"
    )

    lines.append("\nMISSING INFORMATION")
    if triage_result["missing_information"]:
        for item in triage_result["missing_information"]:
            lines.append(f"- {item}")
    else:
        lines.append("- None identified")

    lines.append("\nFACTUAL CONFLICTS")
    if triage_result["conflicts"]:
        for conflict in triage_result["conflicts"]:
            lines.append(f"- {conflict}")
    else:
        lines.append("- None identified")

    lines.append("\nATTORNEY REVIEW FLAGS")
    if triage_result["review_flags"]:
        for flag in triage_result["review_flags"]:
            lines.append(f"- {flag}")
    else:
        lines.append("- None identified")

    lines.append(
        "\nFOR ATTORNEY REVIEW — NOT A LEGAL DETERMINATION"
    )

    return "\n".join(lines)

## End-to-End AI Workflow

The final processing function connects the individual components into one
workflow:

**Client Narrative → AI Extraction → JSON Parsing → Validation →
Deterministic Triage → Attorney Report**

If the AI response cannot be parsed as valid JSON or fails validation,
processing stops and the case is flagged for review rather than allowing
potentially malformed data to continue through the workflow.

In [79]:
def analyze_case(intake_text):
    # Require an intake narrative.
    if not intake_text.strip():
        return "Please enter a client intake narrative."

    # Step 1: AI fact extraction.
    try:
        response = client_ai.responses.create(
            model="gpt-5.6-luna",
            input=[
                {
                    "role": "system",
                    "content": extraction_prompt_v5
                },
                {
                    "role": "user",
                    "content": intake_text
                }
            ]
        )
    except Exception as error:
        return (
            "PROCESSING STOPPED\n\n"
            "The AI extraction request could not be completed.\n\n"
            f"Technical detail: {error}"
        )

    # Step 2: Convert the AI response into structured data.
    try:
        extracted_data = json.loads(response.output_text)
    except json.JSONDecodeError:
        return (
            "PROCESSING STOPPED\n\n"
            "The AI response could not be converted into valid "
            "structured data.\n\n"
            "Attorney review required."
        )

    # Step 3: Validate the structured output.
    is_valid, validation_errors = validate_output(extracted_data)

    if not is_valid:
        error_text = "\n".join(
            f"- {error}" for error in validation_errors
        )

        return (
            "PROCESSING STOPPED\n\n"
            "The AI output failed validation.\n\n"
            f"{error_text}\n\n"
            "Attorney review required."
        )

    # Step 4: Apply deterministic triage rules.
    triage_result = run_triage(extracted_data)

    # Step 5: Generate the attorney-facing report.
    return build_report_text(extracted_data, triage_result)

## Interactive Prototype

The interface below provides a simple demonstration of the complete workflow.

Enter a **synthetic** immigration client intake narrative and select
**Analyze Case**. The application will run the narrative through the AI
extraction, validation, deterministic triage, and reporting pipeline.

For demonstration purposes only. Do not enter confidential client information.

In [80]:
final_app = gr.Interface(
    fn=analyze_case,

    inputs=gr.Textbox(
        lines=14,
        label="Client Intake Narrative",
        placeholder=(
            "Enter a synthetic immigration client intake narrative..."
        )
    ),

    outputs=gr.Textbox(
        lines=24,
        label="Attorney Triage Report"
    ),

    title="Immigration Case Triage Assistant",

    description=(
        "AI-assisted legal workflow prototype that extracts structured facts "
        "from an immigration intake, validates the AI-generated data, "
        "identifies missing information and factual conflicts, applies "
        "deterministic triage rules, and generates an attorney-facing report. "
        "Use synthetic data only."
    ),

    submit_btn="Analyze Case",
    clear_btn="Clear"
)


## Prompt Iteration Findings

### Test Case 4 — Multi-Person Immigration Intake

**Objective:**  
Evaluate whether the model could accurately extract the client's facts when the intake also contained immigration information belonging to another family member.

**Prompt V3 Result:**  
13/16 factual fields correct (81.2%).

**Observed Errors:**
- Entry date was factually correct but returned in an inconsistent date format.
- I-360 priority date was factually correct but returned in an inconsistent date format.
- `asylum_within_one_year` was incorrectly returned as `false` even though the client had not filed asylum.

**Prompt Changes:**  
Prompt V4 added:
- Standardized YYYY-MM-DD date formatting.
- A dependency rule requiring `asylum_within_one_year` to be null when asylum was not filed or timing cannot be determined.
- Explicit entity-attribution instructions to prevent facts belonging to another person from being assigned to the client.

**Prompt V4 Result:**  
16/16 factual fields correct (100%) on the same test case.

**Finding:**  
Targeted prompt changes corrected the identified errors on this test case while maintaining correct attribution of facts between multiple individuals.

**Limitation:**  
Results from a single test case do not establish overall system accuracy. Additional test cases are required to evaluate whether the improvements generalize.

## Prompt V5 Finding

### Test Case 5 — Noisy Multi-Person Narrative

**Prompt V4 Result:**  
15/16 factual fields correct (93.8%).

**Observed Error:**  
The model correctly identified that asylum had been filed but returned `asylum_within_one_year = false` based on an imprecise filing date ("sometime in 2025").

**Prompt Change:**  
Prompt V5 added an asylum timing precision rule requiring sufficiently precise filing information before assigning `true` or `false` to `asylum_within_one_year`.

**Prompt V5 Result:**  
16/16 factual fields correct (100%) on the same test case.

**Finding:**  
The targeted prompt change eliminated the unsupported timing inference while preserving the other correctly extracted fields.

**Limitation:**  
Improvement on one test case does not establish general reliability. The revised prompt should be regression-tested against earlier test cases and additional unseen cases.

## Regression Testing Finding

### Test Case 3 — Contradictory Information Re-tested with Prompt V5

**Purpose:**  
Determine whether changes introduced in Prompt V5 preserved behavior on an earlier contradictory-information test case.

**Result:**  
15/16 factual fields correct (93.8%).

**Conflict Detection:**  
3/3 conflicts detected.

**Regression Identified:**  
`i360_filed` was returned as `true`, while the attorney-defined benchmark expected `null`.

The intake stated that the client initially reported an approved I-360 and later reported that the I-360 was still pending. Both statements imply the existence of an I-360, but neither explicitly states that Form I-360 was filed.

Under the project's conservative extraction policy, converting this implication into `i360_filed = true` constitutes an inference rather than direct fact extraction.

**Finding:**  
Prompt V5 corrected the asylum-timing error identified in Test Case 5 but did not preserve perfect factual-field performance on Test Case 3.

This demonstrates the importance of regression testing: a prompt modification that improves performance on one scenario may alter behavior on a previously tested scenario.

**Next Step:**  
Evaluate Prompt V5 across the broader test set before making additional prompt changes.

In [81]:
def evaluate_case(test_name, expected_output, actual_output):
    correct = 0
    total = 0
    failed_fields = []

    print(test_name)
    print("-" * len(test_name))

    for field in expected_output:

        # Evaluate conflicts separately
        if field == "conflicts":
            continue

        expected = expected_output[field]
        actual = actual_output[field]

        total += 1

        if expected == actual:
            print(field, ": PASS")
            correct += 1
        else:
            print(
                field,
                ": FAIL | Expected:",
                expected,
                "| Actual:",
                actual
            )
            failed_fields.append(field)

    accuracy = (correct / total) * 100

    print("\nRESULT")
    print("------")
    print("Correct factual fields:", correct, "/", total)
    print("Accuracy:", round(accuracy, 1), "%")

    if failed_fields:
        print("Failed fields:", failed_fields)
    else:
        print("Failed fields: None")

    return {
        "test_name": test_name,
        "correct": correct,
        "total": total,
        "accuracy": accuracy,
        "failed_fields": failed_fields
    }

In [82]:
expected_types = {
    "name": (str, type(None)),
    "age": (int, type(None)),
    "country": (str, type(None)),
    "entry_date": (str, type(None)),
    "manner_of_entry": (str, type(None)),
    "in_removal_proceedings": (bool, type(None)),
    "immigration_court": (str, type(None)),
    "next_hearing": (str, type(None)),
    "family_court_order": (bool, type(None)),
    "i360_filed": (bool, type(None)),
    "i360_approved": (bool, type(None)),
    "i360_priority_date": (str, type(None)),
    "deferred_action": (bool, type(None)),
    "asylum_filed": (bool, type(None)),
    "asylum_within_one_year": (bool, type(None)),
    "i485_filed": (bool, type(None)),
    "conflicts": (list,)
}

print("Expected field types:", len(expected_types))

Expected field types: 17


In [83]:
def validate_output(output):
    missing_fields = []
    type_errors = []

    for field in expected_types:

        # Check whether the field exists
        if field not in output:
            missing_fields.append(field)
            continue

        # Check whether the value has the correct type
        value = output[field]

        if not isinstance(value, expected_types[field]):
            type_errors.append(
                {
                    "field": field,
                    "value": value,
                    "actual_type": type(value).__name__
                }
            )

    if missing_fields or type_errors:
        print("OUTPUT VALIDATION: FAIL")

        if missing_fields:
            print("Missing fields:", missing_fields)

        if type_errors:
            print("Type errors:", type_errors)

        return False

    print("OUTPUT VALIDATION: PASS")
    print("All required fields and data types are valid.")

    return True

In [84]:
def validation_gate(output):
    if validate_output(output):
        print("Proceed to attorney triage workflow.")
        return True

    print("Stop: AI output requires review before processing.")
    return False

In [85]:
def display_value(value):
    if value is True:
        return "Yes"

    if value is False:
        return "No"

    if value is None:
        return "Unknown"

    return value

In [86]:
def run_triage_v2(client):
    missing_information = []
    review_flags = []

    # Missing information checks
    if client["manner_of_entry"] is None:
        missing_information.append("Manner of entry")

    if client["next_hearing"] is None:
        missing_information.append(
            "Next immigration court hearing"
        )

    # Attorney review flags
    if (
        client["i360_approved"] is True
        and client["in_removal_proceedings"] is True
    ):
        review_flags.append(
            "Approved I-360 + removal proceedings: "
            "review procedural strategy."
        )

    if (
        client["asylum_filed"] is True
        and client["asylum_within_one_year"] is False
    ):
        review_flags.append(
            "Potential asylum one-year filing issue: "
            "attorney review required."
        )

    if (
        client["i360_approved"] is True
        and client["i485_filed"] is False
    ):
        review_flags.append(
            "I-360 approved but I-485 not filed: "
            "review adjustment eligibility and visa availability."
        )

    return {
        "missing_information": missing_information,
        "review_flags": review_flags,
        "conflicts": client["conflicts"]
    }

In [87]:
!pip -q install gradio

In [88]:
import gradio as gr

print("Gradio loaded successfully.")
print("Version:", gr.__version__)

Gradio loaded successfully.
Version: 6.26.0


In [89]:
def build_report_text(client, triage_result):

    report = []

    report.append("IMMIGRATION CASE TRIAGE REPORT")
    report.append("=" * 50)

    report.append("\nCLIENT")
    report.append("------")
    report.append(
        f"Name: {display_value(client['name'])}"
    )
    report.append(
        f"Age: {display_value(client['age'])}"
    )
    report.append(
        f"Country: {display_value(client['country'])}"
    )
    report.append(
        f"Entry Date: {display_value(client['entry_date'])}"
    )

    report.append("\nIMMIGRATION COURT")
    report.append("-----------------")
    report.append(
        "Removal Proceedings: "
        + str(display_value(
            client["in_removal_proceedings"]
        ))
    )
    report.append(
        "Court: "
        + str(display_value(
            client["immigration_court"]
        ))
    )
    report.append(
        "Next Hearing: "
        + str(display_value(
            client["next_hearing"]
        ))
    )

    report.append("\nSIJS / I-360")
    report.append("------------")
    report.append(
        "Family Court Order: "
        + str(display_value(
            client["family_court_order"]
        ))
    )
    report.append(
        "I-360 Filed: "
        + str(display_value(
            client["i360_filed"]
        ))
    )
    report.append(
        "I-360 Approved: "
        + str(display_value(
            client["i360_approved"]
        ))
    )
    report.append(
        "Priority Date: "
        + str(display_value(
            client["i360_priority_date"]
        ))
    )
    report.append(
        "Deferred Action: "
        + str(display_value(
            client["deferred_action"]
        ))
    )

    report.append("\nASYLUM")
    report.append("------")
    report.append(
        "I-589 Filed: "
        + str(display_value(
            client["asylum_filed"]
        ))
    )
    report.append(
        "Filed Within One Year: "
        + str(display_value(
            client["asylum_within_one_year"]
        ))
    )

    report.append("\nADJUSTMENT OF STATUS")
    report.append("--------------------")
    report.append(
        "I-485 Filed: "
        + str(display_value(
            client["i485_filed"]
        ))
    )

    report.append("\nMISSING INFORMATION")
    report.append("-------------------")

    if triage_result["missing_information"]:
        for item in triage_result["missing_information"]:
            report.append("- " + item)
    else:
        report.append("None identified.")

    report.append("\nFACTUAL CONFLICTS")
    report.append("-----------------")

    if triage_result["conflicts"]:
        for conflict in triage_result["conflicts"]:
            report.append("- " + conflict)
    else:
        report.append("None identified.")

    report.append("\nATTORNEY REVIEW FLAGS")
    report.append("---------------------")

    if triage_result["review_flags"]:
        for flag in triage_result["review_flags"]:
            report.append("- " + flag)
    else:
        report.append("None identified.")

    report.append("\n" + "=" * 50)
    report.append(
        "FOR ATTORNEY REVIEW — NOT A LEGAL DETERMINATION"
    )

    return "\n".join(report)

In [90]:
def analyze_case(intake_text):

    if not intake_text.strip():
        return "Please enter a client intake narrative."

    # Step 1: AI fact extraction
    response = client_ai.responses.create(
        model="gpt-5.6-luna",
        input=[
            {
                "role": "system",
                "content": extraction_prompt_v5
            },
            {
                "role": "user",
                "content": intake_text
            }
        ]
    )

    # Step 2: Convert AI response into Python data
    try:
        extracted_data = json.loads(
            response.output_text
        )
    except json.JSONDecodeError:
        return (
            "PROCESSING STOPPED\n\n"
            "The AI response could not be converted "
            "into valid structured data.\n\n"
            "Attorney review required."
        )

    # Step 3: Validate structure and data types
    if not validate_output(extracted_data):
        return (
            "PROCESSING STOPPED\n\n"
            "The AI output failed validation.\n\n"
            "Attorney review required."
        )

    # Step 4: Run deterministic triage rules
    triage_result = run_triage_v2(
        extracted_data
    )

    # Step 5: Build attorney-facing report
    report = build_report_text(
        extracted_data,
        triage_result
    )

    return report

In [91]:
final_app = gr.Interface(
    fn=analyze_case,

    inputs=gr.Textbox(
        lines=14,
        label="Client Intake Narrative",
        placeholder=(
            "Paste a synthetic immigration "
            "client intake here..."
        )
    ),

    outputs=gr.Textbox(
        lines=24,
        label="Attorney Triage Report"
    ),

    title="Immigration Case Triage Assistant",

    description=(
        "AI-assisted prototype that extracts structured "
        "facts from an immigration intake, validates the "
        "AI output, identifies missing information and "
        "factual conflicts, applies deterministic triage "
        "rules, and generates a report for attorney review. "
        "Use synthetic data only."
    )
)

final_app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a5228a33c0566d9ce4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Prototype Evaluation Summary

### Objective
Build an AI-assisted immigration intake triage prototype that converts
unstructured client narratives into structured information for attorney review.

### Workflow
Client Narrative → AI Fact Extraction → Structured JSON → Validation →
Deterministic Triage Rules → Attorney-Facing Report → Human Review

### Evaluation Approach
Synthetic immigration intake scenarios were created to test common legal-AI
failure modes, including:

- uncertain or incomplete facts
- contradictory client statements
- multiple individuals in one narrative
- incomplete dates
- unsupported factual inference
- malformed structured output
- incorrect data types

### Prompt Iteration
Prompt instructions were revised in response to observed model errors.

Examples included:

- requiring uncertain facts to remain null
- preserving factual contradictions rather than resolving them
- standardizing dates
- preventing facts belonging to another person from being attributed to the client
- requiring sufficiently precise information before determining asylum filing timing

### Test Results
Prompt V5 produced **64 correct factual fields out of 64 evaluated fields (100.0%)** across Test Cases 3–6 in the final reproducible evaluation run.

The unseen sparse-intake Test Case 6 produced 16/16 correct factual fields.

These results apply only to this small synthetic evaluation set and should not be interpreted as general model accuracy.

## Regression Testing

During prompt development, an earlier regression run returned `i360_filed = true` where the attorney-defined benchmark expected `null`. The narrative implied that an I-360 existed but did not explicitly state that Form I-360 had been filed.

This exposed a tension between logical inference and the project's conservative explicit-fact-only extraction policy. The issue was retained as part of the testing history and used to evaluate whether later prompt behavior remained consistent with the attorney-defined benchmark.

### Safety and Quality Controls
The prototype includes:

- required-field validation
- data-type validation
- processing gates for malformed AI output
- explicit representation of unknown information
- factual conflict detection
- separation of AI extraction from deterministic rules
- attorney-review flags rather than automated legal determinations
- human-in-the-loop review

### Limitations
The prototype was evaluated on a small set of synthetic immigration scenarios. The reported 100.0% factual-field accuracy reflects performance only on this limited synthetic evaluation set and should not be interpreted as general accuracy on immigration matters or real-world client files.

The system does not determine eligibility for immigration relief and is not
designed to replace attorney judgment.

Real-world deployment would require substantially broader testing, privacy and
confidentiality controls, security review, monitoring, and additional validation.

### Data
All examples used for development and testing are synthetic. No confidential
client information is included in the project.